In [1]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

try:
    import irksome
except ImportError:
    !python3 -m pip install --no-dependencies git+https://github.com/firedrakeproject/Irksome.git#egg=Irksome
    import irksome

The Navier-Stokes equations
===========================

Irksome also works fine with nonlinear problems and differential algebraic systems:

In [2]:
from firedrake import *  # noqa: F403
from irksome import Dt, RadauIIA, TimeStepper

Here, we'll do the lid-driven cavity at a rather modest Reynolds number.
Because our eventual solver is going to use geometric multigrid, we need to set up a mesh hierarchy:

In [3]:
base_msh = UnitSquareMesh(4, 4)
mh = MeshHierarchy(base_msh, 3)   # 16 x 16 mesh
msh = mh[-1]

Now we set up the rest of the problem, up to semidiscrete form.

In [4]:
V = VectorFunctionSpace(msh, 'CG', 2)
W = FunctionSpace(msh, 'CG', 1)
Z = V * W
up = Function(Z)
u, p = split(up)
v, q = TestFunctions(Z)

Re = Constant(10.0)

t = Constant(0.0)
dt = Constant(1.0 / 16)

F = (inner(Dt(u), v) * dx + inner(dot(grad(u), u), v) * dx
     + 1/Re * inner(grad(u), grad(v)) * dx - inner(p, div(v)) * dx
     - inner(div(u), q) * dx)

bcs = [DirichletBC(Z.sub(0), 0, (1, 2, 3)),
       DirichletBC(Z.sub(0), as_vector([1, 0]), (4,))]

RadauIIA works fine for DAE systems like the Navier-Stokes equations, so we don't have to do anything special to set up the scheme and problem:

In [5]:
num_stages = 2
scheme = RadauIIA(num_stages)
stepper = TimeStepper(F, scheme, t, dt, up, bcs=bcs)


And we can just run it:

In [6]:
while float(t) < 1.0:
    print(f"t: {float(t)}")
    stepper.advance()
    t.assign(t + dt)

t: 0.0
t: 0.0625
t: 0.125
t: 0.1875
t: 0.25
t: 0.3125
t: 0.375
t: 0.4375
t: 0.5


ConvergenceError: Nonlinear solve failed to converge after 50 nonlinear iterations.
Reason:
   DIVERGED_MAX_IT

Now we'll make a pressure plot, which just has corner singularities:

In [ ]:
%config InlineBackend.figure_format = 'svg'

import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (11, 6)

In [ ]:
from firedrake.pyplot import tripcolor, triplot

In [ ]:
u_, p_ = up.subfunctions
p_ -= assemble(p_*dx)
fig, axes = plt.subplots()
collection = tripcolor(up.subfunctions[1], axes=axes, cmap='coolwarm')
fig.colorbar(collection);

Solvers are rather for Navier-Stokes.  The Rana preconditioner tends not to perform as well for convective problems as for the heat equation, and there's no magical black-box algebraic solver, anyway.  Instead, we're going to do a monolithic multigrid scheme based on Vanka relaxation.  This scheme does geometric multigrid with a patch-based relaxation.  For each vertex, we take all velocity degrees of freedom on each cell sharing that vertex (the star) and all pressure degrees of freedom in those cells, omitting the boundary.

The monolithic multigrid scheme is based on the same approach, except that each patch has to contain the degrees of freedom for all of the Runge--Kutta stages.

To control `ASMVankaPC`, we have to tell it to exclude the pressures on the boundary.  In Irksome for mixed problems, we have a "stage major" storage pattern, which means that every other field in our storage is a pressure stage:

In [ ]:
ind_pressure = ",".join([str(2*i+1) for i in range(num_stages)])


And then the rest of the options are like they would be for Vanka

In [ ]:
nonlin_rtol = 1.e-8
nonlin_atol = 1.e-8

solver_params = {
    "mat_type": "aij",
    "snes_type": "newtonls",
    "snes_converged_reason": None,
    "snes_linesearch_type": "l2",
    "ksp_type": "fgmres",
    "ksp_converged_reason": None,
    "ksp_max_it": 30,
    "snes_rtol": nonlin_rtol,
    "snes_atol": nonlin_atol,
    "snes_ksp_ew": None,
    "pc_type": "mg",
    "pc_mg_type": "multiplicative",
    "pc_mg_cycles": "v",
    "mg_levels": {
        "ksp_type": "gmres",
        "ksp_max_it": 3,
        "ksp_convergence_test": "skip",
        "pc_type": "python",
        "pc_python_type": "firedrake.ASMVankaPC",
        "pc_vanka_construct_dim": 0,
        "pc_vanka_sub_sub_pc_type": "lu",
        "pc_vanka_sub_sub_pc_factor_mat_solver_type": "umfpack",
        "pc_vanka_exclude_subspaces": ind_pressure},
    "mg_coarse": {
        "ksp_type": "preonly",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
        "mat_mumps_icntl_14": 200}
}


Now we'll re-run with these parameters, with some PETSc monitors turned on to show that the linear solver just takes a few iterations per Newton step.

In [ ]:
t = Constant(0.0)
dt = Constant(1.0 / 16)
up.assign(0)
stepper = TimeStepper(F, scheme, t, dt, up, bcs=bcs, solver_parameters=solver_params)

while float(t) < 1.0:
    print(f"t: {float(t)}")
    stepper.advance()
    t.assign(t + dt)
